## 6. N-BEATS Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [7]:
import json

from src.dataloader import load_cgm_stats, load_tensors
from src import metrics
from src import train
from src.models.nn_baseline import NBEATSPredictor, HORIZONS

MODEL_NAME = 'NBEATS'
SEED = 42  # change this and re-run for a multi-seed stability comparison
# Built from REPO_ROOT, not the current working directory: Colab's cwd is /content regardless of
# where this notebook file itself lives on the mounted Drive, so a plain relative path like
# Path('results') would silently write to the wrong (ephemeral) place.
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_PATH = EXPERIMENT_DIR / 'results' / f'nbeats_seed{SEED}_results.json'
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoints' / f'nbeats_seed{SEED}'

In [8]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

print('n_train_batches', len(train_loader), 'n_val_batches', len(val_loader),
      'n_test_batches', len(test_loader))

sample_cgm, sample_target = next(iter(train_loader))
print('input batch shape (cgm):', tuple(sample_cgm.shape), sample_cgm.dtype)
print('target batch shape:', tuple(sample_target.shape), sample_target.dtype)

n_train_batches 327 n_val_batches 66 n_test_batches 66
input batch shape (cgm): (256, 24) torch.float32
target batch shape: (256, 5) torch.float32


In [10]:
train.set_seed(SEED)  # before constructing the model -- weight init draws from this
model = NBEATSPredictor(context_length=sample_cgm.shape[1])  # tie to actual data shape, not a
                                                              # hardcoded 24, in case the window
                                                              # size ever changes upstream

model, history = train.train_model(
    model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
    checkpoint_dir=CHECKPOINT_DIR,
)

device: cpu  model: NBEATSPredictor  params: 75,312
epoch   1/50  train_loss=0.4498  val_loss=0.3497  val_rmse(mg/dL)=[12.8, 18.8, 25.7, 29.8, 32.2]
epoch   2/50  train_loss=0.3955  val_loss=0.3372  val_rmse(mg/dL)=[11.8, 18.0, 25.2, 29.4, 32.0]
epoch   3/50  train_loss=0.3888  val_loss=0.3388  val_rmse(mg/dL)=[12.0, 18.0, 25.2, 29.5, 32.1]
epoch   4/50  train_loss=0.3854  val_loss=0.3364  val_rmse(mg/dL)=[11.4, 17.9, 25.2, 29.5, 32.1]
epoch   5/50  train_loss=0.3846  val_loss=0.3382  val_rmse(mg/dL)=[11.6, 17.9, 25.2, 29.5, 32.2]
epoch   6/50  train_loss=0.3834  val_loss=0.3430  val_rmse(mg/dL)=[12.1, 18.2, 25.4, 29.7, 32.3]
epoch   7/50  train_loss=0.3818  val_loss=0.3326  val_rmse(mg/dL)=[11.3, 17.7, 25.1, 29.3, 31.9]
epoch   8/50  train_loss=0.3801  val_loss=0.3386  val_rmse(mg/dL)=[11.5, 17.9, 25.2, 29.6, 32.2]
epoch   9/50  train_loss=0.3792  val_loss=0.3396  val_rmse(mg/dL)=[11.7, 17.9, 25.2, 29.5, 32.3]
epoch  10/50  train_loss=0.3799  val_loss=0.3357  val_rmse(mg/dL)=[11.4, 17

In [11]:
y_pred_val, y_true_val = train.predict(model, val_loader)
y_pred_test, y_true_test = train.predict(model, test_loader)

val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

print('=== Validation (best checkpoint) ===')
print(metrics.format_results_table(val_results))
print()
print('=== Test ===')
print(metrics.format_results_table(test_results))

=== Validation (best checkpoint) ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    11.33 |     7.80 |    12.27 |      99.59
           30 |    17.73 |    11.87 |    19.96 |      99.28
           60 |    25.12 |    16.69 |    29.63 |      98.74
           90 |    29.31 |    19.67 |    35.27 |      98.25
          120 |    31.91 |    22.05 |    38.56 |      97.97

=== Test ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    11.85 |     8.23 |    12.95 |      99.73
           30 |    18.84 |    12.58 |    21.68 |      99.52
           60 |    27.28 |    18.12 |    33.04 |      99.01
           90 |    32.37 |    21.68 |    40.09 |      98.65
          120 |    35.24 |    24.03 |    43.80 |      98.38


In [12]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seed': SEED,
        'val_results': val_results,
        'test_results': test_results,
        'n_epochs_trained': len(history['train_loss']),
    }, f, indent=2)

print(f'saved to {RESULTS_PATH}')
print(f'checkpoints (best.pt / final.pt / history.json) saved to {CHECKPOINT_DIR}')

saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/results/nbeats_seed42_results.json
checkpoints (best.pt / final.pt / history.json) saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/checkpoints/nbeats_seed42
